# Student Distillation: German Customer Support (QLoRA)

Trains `Llama-3.2-1B-Instruct` on teacher-generated (prompt, response) pairs via **sequence-level knowledge distillation**.   
The student learns the teacher's domain style and response format without ever seeing the teacher's internal probability distributions.   

**Pipeline position:** 01 Fine-Tuning → 02 Data Generation → `[03 Student Distillation]` → 04 Evaluation   

**Strong GPU required.** This notebook was developed on a Kaggle T4 (16 GB VRAM).   
[![Open Notebook in Kaggle](https://img.shields.io/badge/Open%20Notebook%20in-Kaggle-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white)](https://www.kaggle.com/code/dennisfeyerabend/03-student-distillation)

## 1. Setup

Install dependencies and check GPU.  
Local users: skip the pip cell — install via `pip install -r requirements.txt` instead.  
Checks if CUDA capable GPU is available

In [1]:
%%capture
!pip install -q --upgrade unsloth trl datasets transformers peft bitsandbytes accelerate huggingface_hub python-dotenv

In [2]:
!nvidia-smi
import torch
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sat May 23 01:19:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Authenticate

**Kaggle:** add `HF_TOKEN` via *Add-ons → Secrets*.   
**Local:** create a `.env` file with `HF_TOKEN=your_token`.   

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env locally, no-op on Kaggle

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except ImportError:
    print("kaggle_secrets not available — using .env fallback.")
except Exception as e:
    print(f"Kaggle Secrets found but could not load HF_TOKEN: {e}")

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    print(f"HF_TOKEN set. ({HF_TOKEN[:4]}...{HF_TOKEN[-4:]})")
else:
    print("WARNING: HF_TOKEN is not set. The save cell will fail.")

HF_TOKEN loaded from Kaggle Secrets.
HF_TOKEN set. (hf_R...nFKm)


## 3. Load Student Base Model

**Why `Llama-3.2-1B-Instruct`?**
- 7× smaller than the teacher (7B vs 1B) — a stronger compression ratio, and a more realistic deployment target
- Multilingual pretraining covers German well despite not being a German specialist
- Fits in ~0.8 GB VRAM in 4-bit — ample headroom for training on a T4
- Native Unsloth support (Llama-family kernels)
- Llama 3.2 Community License — commercial use permitted 

In [4]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None           # auto-detect: float16 on T4, bfloat16 on Ampere+
LOAD_IN_4BIT = True    # QLoRA: quantize base weights to 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"Loaded: {model.config._name_or_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


Loaded: unsloth/Llama-3.2-1B-Instruct-bnb-4bit


## 4. Apply QLoRA Adapter

Same LoRA configuration as the teacher (Notebook 01) — `r=16`, all attention and MLP projections targeted.   
The adapter is initialised from scratch; the base weights remain frozen.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    max_seq_length = MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Unsloth 2026.5.6 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Trainable parameters: 11,272,192 / 760,547,328 (1.48%)


## 5. Load Distillation Dataset

The dataset is the 232 (prompt, response) pairs generated by the teacher in Notebook 02.   
It is stored alongside the LoRA adapter on Hugging Face Hub and downloaded directly from there — no local file needed.   

In [6]:
import json
from huggingface_hub import hf_hub_download

DATA_REPO = "Feyerade/german-support-leollm-lora-adapter"

file_path = hf_hub_download(
    repo_id=DATA_REPO,
    filename="teacher_generated_data.json",
    token=HF_TOKEN,
)

with open(file_path, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} teacher-generated examples")
print(f"\nExample entry:")
print(f"  instruction: {raw_data[0]['instruction'][:150]}")
print(f"  output:      {raw_data[0]['output'][:300]}...")

teacher_generated_data.json:   0%|          | 0.00/124k [00:00<?, ?B/s]

Loaded 232 teacher-generated examples

Example entry:
  instruction: Meine Lieferung sollte gestern ankommen, aber im Tracking steht immer noch 'in Bearbeitung'.
  output:      Das verstehe ich — das ist aergerlich! Hier ist, was ichet:

1. Kontaktieren Sie den Verkaeufer mit Ihrer Sendungsverfolgungsnummer.
2. Fragen Sie nach dem Status Ihrer Lieferung und verlangen Sie eine endgueltige Antwort.
3. Notieren Sie sich diese Informationen und teilen Sie dem Verkaeufer mit, d...


## 6. Format Dataset

The goal of this project is style distillation: teaching the student to respond in the same structured, professional format as the teacher — numbered steps, polite phrasing, concise answers. That format is the main thing being transferred here, not factual knowledge.

For that transfer to work, the training data must use the exact same prompt structure the student will use when generating responses.   
Llama 3.2 uses a header-based chat format with clearly labelled roles (`system`, `user`, `assistant`). `apply_chat_template` handles the formatting automatically:
    
    <|start_header_id|>system<|end_header_id|>
    Du bist ein professioneller Kundenservice-Mitarbeiter ...
    <|eot_id|><|start_header_id|>user<|end_header_id|>
    Mein Paket ist nicht angekommen.
    <|eot_id|><|start_header_id|>assistant<|end_header_id|>
    Das verstehe ich — lassen Sie mich das prüfen ...

The system prompt is identical to the one used in Notebook 02 when the teacher generated the responses. This is important: the student is trained on data produced under the exact same instructions it will operate under — so the response style it learns is directly the style it needs to reproduce.

In [7]:
import random
from datasets import Dataset

SYSTEM_PROMPT = (
    "Du bist ein professioneller Kundenservice-Mitarbeiter. "
    "Antworte auf Deutsch und halte dich strikt an folgendes Format:\n"
    "Beginne mit einer kurzen Empathie- oder Begruessungsformel (z.B. 'Das tut mir leid', 'Vielen Dank fuer Ihre Anfrage').\n"
    "Gib deine Instruktionen/Empfehlungen als nummerierte Schritte oder Aufzaehlung — nicht als Fliestext.\n"
    "Schliesse mit einem Angebot zur weiteren Hilfe ab (z.B. 'Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung').\n"
    "Sowohl der einleitende- als auch der abschließende Satz dürfen nicht Teil einer Aufzählung sein.\n"
    "Verwende durchgehend eine professionelle, hoefliche Sprache (Sie-Form).\n"
    "Halte deine Antworten unter 150 Woertern."
)

def format_example(example, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted = [{"text": format_example(ex, tokenizer)} for ex in raw_data]
dataset = Dataset.from_list(formatted)

idx = random.randint(0, len(dataset) - 1)
print(f"Sample (index {idx}):\n")
print(dataset[idx]["text"])

Sample (index 163):

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 May 2026

Du bist ein professioneller Kundenservice-Mitarbeiter. Antworte auf Deutsch und halte dich strikt an folgendes Format:
Beginne mit einer kurzen Empathie- oder Begruessungsformel (z.B. 'Das tut mir leid', 'Vielen Dank fuer Ihre Anfrage').
Gib deine Instruktionen/Empfehlungen als nummerierte Schritte oder Aufzaehlung — nicht als Fliestext.
Schliesse mit einem Angebot zur weiteren Hilfe ab (z.B. 'Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung').
Sowohl der einleitende- als auch der abschließende Satz dürfen nicht Teil einer Aufzählung sein.
Verwende durchgehend eine professionelle, hoefliche Sprache (Sie-Form).
Halte deine Antworten unter 150 Woertern.<|eot_id|><|start_header_id|>user<|end_header_id|>

Ich bekomme keine Updates zum Versand.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Das verstehe ich — das ist frustrierend! 

## 7. Train

Standard SFT on (prompt, teacher-response) pairs — no access to teacher logits required.   
This is **sequence-level knowledge distillation** (Kim & Rush, 2016): the student mimics the teacher's outputs rather than its internal probability distributions.   

Key hyperparameters:
- `per_device_train_batch_size=2` + `gradient_accumulation_steps=4` → effective batch size of 8   
- `max_steps=200` — ~6 epochs over 232 examples; loss convergence verified empirically   
- `optim="adamw_8bit"` — memory-efficient optimizer from bitsandbytes
- `report_to="none"` — disables W&B/HF logging

In [8]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=200,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

print(f"\nSteps:         {trainer_stats.global_step}")
print(f"Average training loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime:       {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Samples/sec:   {trainer_stats.metrics['train_samples_per_second']:.2f}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/232 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 232 | Num Epochs = 7 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.487822
20,0.927736
30,0.633742
40,0.540445
50,0.563504
60,0.504173
70,0.471931
80,0.444316
90,0.420359
100,0.391801


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.



Steps:         200
Average training loss: 0.4991
Runtime:       321s
Samples/sec:   4.98


## 8. Qualitative Evaluation

Sanity check on the same held-out queries used in Notebook 01.   
This lets you compare student and teacher outputs side-by-side without running Notebook 04.   
Systematic benchmarking (BERTScore, latency, VRAM) is in Notebook 04.   

In [9]:
import transformers
import warnings
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

FastLanguageModel.for_inference(model)

test_queries = [
    "Mein Paket ist seit 2 Wochen nicht angekommen.",
    "Kann ich auch per Rechnung bezahlen?",
    "Ich moechte meine Adresse aendern.",
    "Das Produkt entspricht nicht der Beschreibung.",
    "Gibt es das auch in Groesse XL?",
]

for query in test_queries:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": query},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    print("=" * 60)
    print(f"Query:    {query}")
    print("=" * 60)
    print(f"Response: {response}")

Query:    Mein Paket ist seit 2 Wochen nicht angekommen.
Response: Das verstehe ich — das ist aergerlich! Hier ist, was ichet:
1. Kontaktieren Sie den Verkaeufer: Oeffnen Sie eine E-Mail oder einen Chat mit dem Verkaeufer des Pakets.
2. Fragen stellen: Frage nach dem Absender, dem Empfaenger und dem Verlauf des Pakets.
3. Ressourcen nutzen: Nutzen Sie die Ressourcen des Verkaeuers, wie z.B. die Paketdienst-App, um weitere Informationen zu erhalten.
4. Hilfe anbieten: Wenn Sie weitere Fragen haben, stehe ich Ihnen gerne zur Verfuegung. Bitte entschuldigen Sie die Unannehmlichkeiten — bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung.
Query:    Kann ich auch per Rechnung bezahlen?
Response: 1. Vielen Dank fuer Ihre Anfrage
2. Die Preise auf unserer Website sind mit dem aktuellen Preis gefeischt
3. Wir senden Ihnen eine Rechnung per Post
4. Bitte kontaktieren Sie uns mit Ihrer Rechnungsnummer, um Ihre Zahlung zu bezahlen
5. Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung
Q

## 9. Push to Hugging Face Hub

Saves the merged student model (base + LoRA weights) and pushes to Hugging Face Hub.   
Switch `if False` to `if True` to activate a section.   

**Kaggle:** add `HF_TOKEN` via *Add-ons → Secrets*.   
**Local:** set `HF_TOKEN` as an environment variable.   

The merged model is pushed as a standalone checkpoint — no adapter files needed at inference time.   

In [11]:
REPO_ID = "Feyerade/german-support-llama-1b-distilled"

# Save merged model locally
if False:
    model.save_pretrained_merged(
        save_directory="german_support_student_merged",
        tokenizer=tokenizer,
        save_method="merged_16bit",
    )
    print("Merged student model saved locally!")

# Push merged model directly to Hugging Face Hub
if True:
    model.push_to_hub_merged(
        REPO_ID,
        tokenizer=tokenizer,
        save_method="merged_16bit",
        token=HF_TOKEN,
    )
    print(f"Pushed to: https://huggingface.co/{REPO_ID}")

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:40<00:00, 40.34s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/Feyerade/german-support-llama-1b-distilled`
Pushed to: https://huggingface.co/Feyerade/german-support-llama-1b-distilled
